# Data Loader

Load and flatten the param sweep JSON files into a tidy table.

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path("binpacking/data_analysis/binpacking_settings/param_sweep")

def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def flatten_record(data: dict, path: Path) -> dict:
    agg = data.get("aggregate", {})
    problem = data.get("problem", {})
    return {
        "file": path.name,
        "path": str(path),
        "scenario": data.get("scenario"),
        "scenario_description": data.get("scenario_description"),
        "pipeline": data.get("pipeline"),
        "offline_solver": data.get("offline_solver"),
        "online_policy": data.get("online_policy"),
        "seed_count": data.get("seed_count"),
        "total_objective_mean": data.get("total_objective_mean", agg.get("total_objective_mean")),
        "offline_objective_mean": agg.get("offline_objective_mean"),
        "online_objective_mean": agg.get("online_objective_mean"),
        "offline_runtime_mean": agg.get("offline_runtime_mean"),
        "online_runtime_mean": agg.get("online_runtime_mean"),
        "runtime_mean": agg.get("runtime_mean"),
        "offline_failures": agg.get("offline_failures"),
        "online_failures": agg.get("online_failures"),
        "failures": agg.get("failures"),
        "N": problem.get("N"),
        "M_off": problem.get("M_off"),
        "M_on": problem.get("M_on"),
        "dimensions": problem.get("dimensions"),
    }

def load_results(base_dir: Path = BASE_DIR) -> list[dict]:
    records = []
    for path in sorted(base_dir.rglob("*.json")):
        data = read_json(path)
        records.append(flatten_record(data, path))
    return records

records = load_results()
len(records)


In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(records)
    df_eval = df[df["pipeline"].notna()].copy()
    df_opt = df[df["pipeline"].isna()].copy()
    df.head()
except ImportError:
    df = None
    df_eval = None
    df_opt = None
    print("pandas not available; using `records` list of dicts.")
